<a href="https://colab.research.google.com/github/kpramod21/AI_UseCases_Projects/blob/main/Project_1_Banking_customer_support_ai_agent_student_workbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone Project: Banking Customer Support AI Agent

## Student Workbook

In this capstone, you will design and build a simple multi-agent customer support assistant for a banking scenario.

The goal is not to write perfect production code. The goal is to practice the AI engineering workflow:

- Understand the problem
- Break the system into components
- Design prompts for each component
- Generate and revise code
- Test the workflow
- Evaluate what worked and what failed

You may use ChatGPT, Gemini, Claude, or another AI assistant to help generate code. Your job is to guide the assistant with clear prompts, test the output, and improve it.

## 1. Understand the Problem

Your assistant should handle three kinds of customer messages:

1. **Positive Feedback**  
   Example: "Thanks for resolving my credit card issue."

2. **Negative Feedback**  
   Example: "My debit card replacement still hasn't arrived."

3. **Ticket Query**  
   Example: "Can you check the status of ticket 650932?"

The system should:

- Classify the message
- Route it to the correct agent
- Generate a response
- Create tickets for negative feedback
- Look up ticket status for queries

### Discussion Questions

Before writing code, answer these:

1. What are the main tasks this system must perform?
2. Which tasks require AI?
3. Which tasks can be solved with regular Python code?
4. Why might multiple agents be useful here?

In [1]:
# Write brief answers as comments.
# 1. Figure out what policies are there and how they work. From there, put the policies into a RAG for the system to query.
# 2. CLASSIFIER: Message classification: classify incoming messages as positive, negative, query. (AI yes! User sentiment)
# 3. Router -- route messages to the appropriate agent based on the classification.  (AI no! Rule-based) if negative -> ticketing agent, if query -> query response agent, if positive -> thank you agent.
# 4. Ticketing agent -- create a ticket for negative messages, and generate appropriate response.  (AI yes, for response generation, but ticket creation can be rule-based)
# 5. Query response agent -- look up ticket status for query messages. (Ticket status lookup is rule based, but response generation can be AI, or not.)
# 6. Thank you agent -- send thank you messages for positive feedback. (AI or rule-based depending on how personalized we want the responses to be.)
# 7?. (allow retry in case of failure -- flag for follow-up.)
# 8?. Logging agent -- log all interactions for future reference and analysis.


# How do we handle FAQs?
# -> 1. From user question, compute similarity with existing FAQs using a vector database (e.g., FAISS), and return most relevant FAQ (if similarity above threshold) as response.
# -> 2. From user question, build a RAG with policy and other relevant information, and use that to generate a response.
# -> In both cases, we'd need some type of judge agent to OK the answer.

## 2. Proposed Architecture

```text
User Message
      |
      v
Classifier Agent
      |
      +-----------------------+
      |                       |
      v                       v
Feedback Agent           Query Agent
      |                       |
      +-----------+-----------+
                  |
                  v
             Ticket Store
```

### Agent Responsibilities

| Agent | Responsibility |
|---|---|
| Classifier Agent | Decide whether the message is positive feedback, negative feedback, or a query |
| Feedback Agent | Respond to feedback and create new tickets for negative feedback |
| Query Agent | Extract a ticket number and return its status |
| Orchestrator | Connect the agents and route the message |

### Architecture Reflection

1. What is the responsibility of each agent?
2. What happens if the classifier makes a mistake?
3. Could this be built as one big prompt? What would be the downside?

In [2]:
# Architecture notes

## 3. Setup

Keep setup simple. You can use only Python at first. Add an LLM only where it helps.

Suggested imports:

In [3]:
import re
import random
import json
from pathlib import Path

## 4. Build the Classifier Agent

### Goal

Classify a customer message into exactly one of these categories:

- `Positive Feedback`
- `Negative Feedback`
- `Query`

You may use:

- rule-based logic
- an LLM prompt
- a hybrid approach

### Better Prompt

Write a short improved prompt. Do not make it long. Make it clear.

In [4]:
better_classifier_prompt = """
Create a Python ClassifierAgent.

Classify customer messages into exactly one category:
- Positive Feedback
- Negative Feedback
- Query

Return only the category name.
Use OpenAI API to classify the message based on keywords and context.

Examples:
"Thanks for your help" -> Positive Feedback
"My issue is still unresolved" -> Negative Feedback
"What is the status of ticket 123456?" -> Query
"""

In [5]:
from openai import OpenAI
#from dotenv import load_dotenv
#load_dotenv()

from google.colab import userdata

# Removed: import openai (as it conflicts with the new API style where OpenAI is a client)

class ClassifierAgent:
    """
    Classifies customer messages into one of the following categories:
    - Positive Feedback
    - Negative Feedback
    - Query
    """

    def __init__(self, model="gpt-3.5-turbo"):
        self.model = model
        # Retrieve the API key from Colab secrets and pass it to the OpenAI client
        api_key = userdata.get('OPENAI_API_KEY')
        self.client = OpenAI(api_key=api_key) # Initialize the OpenAI client with the API key

    def classify(self, message: str) -> str:
        """
        Classify the customer message.

        Parameters:
            message (str): Customer input message.

        Returns:
            str: One of:
                 - Positive Feedback
                 - Negative Feedback
                 - Query
        """

        prompt = f"""
You are a banking customer support message classifier.

Classify the customer message into exactly one category:
- Positive Feedback
- Negative Feedback
- Query

Definitions:
- Positive Feedback: Appreciation, gratitude, praise, or satisfaction.
- Negative Feedback: Complaints, dissatisfaction, unresolved issues, frustration, or service problems.
- Query: Requests for information, ticket status checks, or questions.

Return ONLY the category name.
Do not provide explanations.

Examples:
"Thanks for your help" -> Positive Feedback
"My issue is still unresolved" -> Negative Feedback
"What is the status of ticket 123456?" -> Query

Customer Message:
"{message}"
"""

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        category = response.choices[0].message.content.strip()

        valid_categories = [
            "Positive Feedback",
            "Negative Feedback",
            "Query"
        ]

        if category not in valid_categories:
            return "Negative Feedback"

        return category

### Generate or Write Your Classifier Code

Use your improved prompt with an AI assistant, then paste or write the generated code below.

## 5. Test the Classifier

Try the provided test cases and add your own.

In [6]:
classifier = ClassifierAgent()

test_messages = [
    "Thanks for your help",
    "Thanks for your help, but my issue is still unresolved",
    "What is the status of ticket 123456?",
    "Thanks, but my issue still is not fixed.",
    "Yeah, this would be great support if I lived in opposite-land. You guys really stink!"
]

for msg in test_messages:
    print(msg, "->", classifier.classify(msg))

Thanks for your help -> Positive Feedback
Thanks for your help, but my issue is still unresolved -> Negative Feedback
What is the status of ticket 123456? -> Query
Thanks, but my issue still is not fixed. -> Negative Feedback
Yeah, this would be great support if I lived in opposite-land. You guys really stink! -> Negative Feedback


### Classifier Test Notes

Record what worked and what failed.

| Message | Expected | Actual | Notes |
|---|---|---|---|
| Thanks for your help | Positive Feedback | | |
| My issue is still unresolved | Negative Feedback | | |
| What is the status of ticket 123456? | Query | | |
| Thanks, but my issue still is not fixed. | Discuss | | |

## 6. Build the Feedback Agent

### Goal

The Feedback Agent handles both positive and negative feedback.

For positive feedback:

- Generate a brief professional thank-you message

For negative feedback:

- Generate a unique 6-digit ticket number
- Save the ticket with status `Unresolved`
- Return an empathetic response with the ticket number

### Better Prompt

Here is a better but still short version.

In [7]:
better_feedback_prompt = """
Create a Python FeedbackAgent.

For Positive Feedback:
- Return a warm professional thank-you response
- Keep it to 1-2 sentences
- Do not invent account details

For Negative Feedback:
- Generate a unique 6-digit ticket number
- Save the ticket with status "Unresolved"
- Return an empathetic response that includes the ticket number

Use information from the message to personalize the response, but do not invent any account details or specific information that is not provided.
Use OpenAI API to generate the responses based on the feedback type and customer name.

In responses should be warm and professional.
"""

### Ticket Store

For this capstone, you can use a simple file or dictionary as the ticket store.

A real system would use a database. For learning, simple storage is enough.

In [8]:
TICKET_FILE = Path("tickets.json")

def load_tickets() -> dict:
    if not TICKET_FILE.exists():
        return {}
    return json.loads(TICKET_FILE.read_text())

def save_tickets(tickets: dict) -> None:
    TICKET_FILE.write_text(json.dumps(tickets, indent=2))

# Seed a sample ticket for testing
sample_tickets = load_tickets()
sample_tickets.setdefault("123456", {"status": "Resolved", "message": "Sample ticket"})
save_tickets(sample_tickets)

### Generate or Write Your Feedback Agent Code

In [9]:
from openai import OpenAI
import random
import json
from pathlib import Path
from google.colab import userdata

# Initialize the OpenAI client with the API key from userdata
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))


class FeedbackAgent:
    def __init__(self, ticket_file: str = "tickets.json"):
        self.ticket_file = Path(ticket_file)
        self._initialize_ticket_file()

    def _initialize_ticket_file(self):
        if not self.ticket_file.exists():
            self.ticket_file.write_text("[]")

    def _load_tickets(self) -> list[dict]:
        return json.loads(self.ticket_file.read_text())

    def _save_tickets(self, tickets: list[dict]) -> None:
        self.ticket_file.write_text(json.dumps(tickets, indent=2))

    def _generate_unique_ticket_number(self) -> str:
        tickets = self._load_tickets()

        existing_ticket_numbers = {
            ticket["ticket_number"]
            for ticket in tickets
            if isinstance(ticket, dict) and "ticket_number" in ticket
        }

        while True:
            ticket_number = str(random.randint(100000, 999999))
            if ticket_number not in existing_ticket_numbers:
                return ticket_number

    def _create_ticket(self, customer_name: str, message: str) -> str:
        ticket_number = self._generate_unique_ticket_number()

        tickets = self._load_tickets()
        tickets.append({
            "ticket_number": ticket_number,
            "customer_name": customer_name,
            "message": message,
            "status": "Unresolved"
        })
        self._save_tickets(tickets)

        return ticket_number

    def _generate_response(
        self,
        feedback_type: str,
        customer_name: str,
        message: str,
        ticket_number: str | None = None
    ) -> str:
        prompt = f"""
You are a banking customer support assistant.

Generate a response for this customer.

Feedback Type: {feedback_type}
Customer Name: {customer_name}
Customer Message: {message}
Ticket Number: {ticket_number if ticket_number else "None"}

Requirements:
- Generate a warm and professional response
- Keep the response to 1-2 sentences.
- Do not invent account details.
- If a ticket number is provided, include it.
"""

        response = client.responses.create(
            model="gpt-4.1-mini",
            input=prompt,
            temperature=0.7,
        )

        return response.output_text.strip()

    def handle_feedback(
        self,
        feedback_type: str,
        message: str,
        customer_name: str = "Customer"
    ) -> str:
        if feedback_type == "Positive Feedback":
            return self._generate_response(
                feedback_type=feedback_type,
                customer_name=customer_name,
                message=message
            )

        if feedback_type == "Negative Feedback":
            ticket_number = self._create_ticket(
                customer_name=customer_name,
                message=message
            )

            return self._generate_response(
                feedback_type=feedback_type,
                customer_name=customer_name,
                message=message,
                ticket_number=ticket_number
            )

        return "Invalid feedback type."

## 7. Test the Feedback Agent

In [10]:
from pathlib import Path

Path("tickets.json").write_text("[]")

classifier = ClassifierAgent()
feedback_agent = FeedbackAgent()

test_messages = [
    "Thanks for your help",
    "Thanks for your help, but my issue is still unresolved",
    "What is the status of ticket 123456?",
    "Thanks, but my issue still is not fixed.",
    "Yeah, this would be great support if I lived in opposite-land. You guys really stink!"
]

for msg in test_messages:
    feedback_type = classifier.classify(msg)
    print(msg)
    print(feedback_agent.handle_feedback(feedback_type, msg))
    print("\n\n")
#print(load_tickets())

Thanks for your help
Dear Customer, 

Thank you for your kind words! We're always here to help whenever you need us.



Thanks for your help, but my issue is still unresolved
Dear Customer, thank you for your patience, and I apologize that your issue is still unresolved. We are actively working on ticket number 946650 and will update you as soon as possible.



What is the status of ticket 123456?
Invalid feedback type.



Thanks, but my issue still is not fixed.
Dear Customer, thank you for your patience. We apologize that your issue is still unresolved and will prioritize ticket number 655434 to ensure a swift resolution.



Yeah, this would be great support if I lived in opposite-land. You guys really stink!
Dear Customer, we’re sorry to hear about your experience and appreciate your feedback. Please rest assured that we are committed to improving our service and will review ticket number 665042 promptly to address your concerns.





### Feedback Agent Test Notes

1. Did positive feedback return a useful response?
2. Did negative feedback create a ticket?
3. Was the ticket saved correctly?
4. Was the response empathetic and clear?

In [11]:
# Yes, positive feedback return a useful response
# Yes, negative feedback create a ticket
# Yes, the ticket saved correctly
# Yes, the response empathetic and clear

## 8. Build the Query Agent

### Goal

The Query Agent should:

- Extract a 6-digit ticket number from the customer message
- Look up the ticket
- Return the ticket status
- Handle missing or invalid ticket numbers

### Better Prompt

In [12]:
better_query_prompt = """
Create a Python QueryAgent.

Requirements:
- Extract a 6-digit ticket number from the customer message
- Look up the ticket in the ticket store
- If found, return: "Your ticket #[TicketNumber] is currently marked as: [Status]."
- If not found, return: "Ticket #[TicketNumber] could not be found."
- If no ticket number is provided, ask the customer to provide a ticket number
"""

### Generate or Write Your Query Agent Code

In [13]:
# Query Agent

import json
import re
import os


class QueryAgent:
    """
    Handles customer ticket status queries.

    Responsibilities:
    - Extract a 6-digit ticket number from the customer message.
    - Look up the ticket in tickets.json.
    - Return the current ticket status if found.
    - Inform the customer if the ticket cannot be found.
    - Ask for a ticket number if none is provided.
    """

    def __init__(self, tickets_file="tickets.json"):
        self.tickets_file = tickets_file

        # Create an empty tickets file if it doesn't exist
        if not os.path.exists(self.tickets_file):
            with open(self.tickets_file, "w") as file:
                json.dump([], file, indent=4)

    def extract_ticket_number(self, message):
        """
        Extract a 6-digit ticket number from the customer message.

        Returns:
            str: Ticket number if found, otherwise None.
        """
        match = re.search(r"\b(\d{6})\b", message)

        if match:
            return match.group(1)

        return None

    def get_ticket_status(self, ticket_number):
        """
        Retrieve the status of the specified ticket.

        Returns:
            str: Ticket status if found, otherwise None.
        """
        try:
            with open(self.tickets_file, "r") as file:
                tickets = json.load(file)

            for ticket in tickets:
                if str(ticket.get("ticket_id")) == ticket_number:
                    return ticket.get("status", "Unknown")

        except (FileNotFoundError, json.JSONDecodeError):
            return None

        return None

    def handle_query(self, message):
        """
        Process the customer query.

        Returns:
            str: Appropriate response based on the query result.
        """
        ticket_number = self.extract_ticket_number(message)

        if not ticket_number:
            return (
                "Could you please provide your 6-digit ticket number "
                "so that I can check the status for you?"
            )

        status = self.get_ticket_status(ticket_number)

        if status:
            return (
                f"Your ticket #{ticket_number} is currently "
                f"marked as: {status}."
            )

        return f"Ticket #{ticket_number} could not be found."

## 9. Test the Query Agent

In [14]:
query_agent = QueryAgent()
# These tests do not cover the case where something was mis-routed.
query_tests = [
    "What is the status of ticket 123456?",
    "Can you check ticket #999999?",
    "Can you check my ticket?",
]

for msg in query_tests:
    print(msg)
    print(query_agent.handle_query(msg))
    print()

What is the status of ticket 123456?
Ticket #123456 could not be found.

Can you check ticket #999999?
Ticket #999999 could not be found.

Can you check my ticket?
Could you please provide your 6-digit ticket number so that I can check the status for you?



### Query Agent Test Notes

| Message | Expected Behavior | Actual Behavior | Notes |
|---|---|---|---|
| What is the status of ticket 123456? | Return status | | |
| Can you check ticket #999999? | Ticket not found | | |
| Can you check my ticket? | Ask for ticket number | | |

## 10. Build the Orchestrator

### Goal

The orchestrator connects the agents.

It should:

1. Receive a customer message
2. Classify the message
3. Route the message to the correct agent
4. Return the final response

### Better Prompt

In [15]:
better_orchestrator_prompt = """
Create a Python OrchestratorAgent.


Workflow:
1. Receive customer message
2. Call ClassifierAgent.classify(message)
3. If category is Positive Feedback or Negative Feedback, call FeedbackAgent.handle_feedback(category,message)
4. If category is Query, call QueryAgent.handle_query(message)
5. Return the final response

Use clear class interfaces and handle unexpected classifier output gracefully.
"""

### Generate or Write Your Orchestrator Code

-- First Code is for LOG FILE
-- Then use it in Orchestrator

In [16]:
import json
from datetime import datetime


class Logger:

    LOG_FILE = "logs.json"

    @classmethod
    def log(
        cls,
        message,
        classification,
        route,
        response
    ):

        try:
            with open(cls.LOG_FILE, "r") as f:
                logs = json.load(f)

        except:
            logs = []

        logs.append({
            "timestamp": datetime.now().isoformat(),
            "message": message,
            "classification": classification,
            "route": route,
            "response": response
        })

        with open(cls.LOG_FILE, "w") as f:
            json.dump(logs, f, indent=4)

In [17]:
class OrchestratorAgent:
    def __init__(self, classifier, feedback_agent, query_agent):
        self.classifier = classifier
        self.feedback_agent = feedback_agent
        self.query_agent = query_agent

    def handle_message(self, message: str) -> dict:
        category = self.classifier.classify(message)
        response_text = ""
        ticket_number = None
        route = category # For display purposes

        if category in ["Positive Feedback", "Negative Feedback"]:
            response_text = self.feedback_agent.handle_feedback(category, message)
            if category == "Negative Feedback":
                # Attempt to extract ticket number if generated by feedback agent
                import re
                match = re.search(r'ticket number (\d{6})', response_text) # Corrected regex for ticket number extraction
                if match:
                    ticket_number = match.group(1)
        elif category == "Query":
            response_text = self.query_agent.handle_query(message)
        else:
            response_text = (
                "I'm sorry, I could not determine how to handle your request. "
                "Please rephrase your message."
            )
            route = "Unknown"

        # Log the interaction after all variables are defined
        Logger.log(
            message=message,
            classification=category,
            route=route,
            response=response_text
        )

        return {"classification": category, "route": route, "response": response_text, "ticket_number": ticket_number}

In [20]:
import json
import pandas as pd

try:
    with open("logs.json", "r") as f:
        logs = json.load(f)
except FileNotFoundError:
    logs = [] # If file not found, initialize logs as an empty list

# Display as a table
if logs:
    df_logs = pd.DataFrame(logs)
    print("Logs:")
    print(df_logs)
else:
    print("No logs found yet. Run the orchestrator with test messages to generate logs.")

No logs found yet. Run the orchestrator with test messages to generate logs.


## 11. End-to-End Testing

Now test the whole workflow.

In [21]:
orchestrator = OrchestratorAgent(
    classifier=ClassifierAgent(),
    feedback_agent=FeedbackAgent(),
    query_agent=QueryAgent(),
)

end_to_end_tests = [
    "Thanks for resolving my issue.",
    "My debit card still hasn't arrived.",
    "What is the status of ticket 123456?",
    "Thanks, but my issue still is not fixed.",
]

for msg in end_to_end_tests:
    print("User:", msg)
    print("Assistant:", orchestrator.handle_message(msg))
    print("-" * 60)

User: Thanks for resolving my issue.
Assistant: {'classification': 'Positive Feedback', 'route': 'Positive Feedback', 'response': "Dear Customer, thank you for your kind feedback! We're glad we could resolve your issue and are here to assist you anytime.", 'ticket_number': None}
------------------------------------------------------------
User: My debit card still hasn't arrived.
Assistant: {'classification': 'Negative Feedback', 'route': 'Negative Feedback', 'response': 'Dear Customer, we apologize for the delay in receiving your debit card and are looking into this issue promptly. Please rest assured that we are working to resolve it and will update you shortly; your ticket number is 823216.', 'ticket_number': None}
------------------------------------------------------------
User: What is the status of ticket 123456?
Assistant: {'classification': 'Query', 'route': 'Query', 'response': 'Ticket #123456 could not be found.', 'ticket_number': None}
--------------------------------------

### End-to-End Test Notes

| Input | Expected Route | Actual Route | Response Quality Notes |
|---|---|---|---|
| Thanks for resolving my issue. | Positive Feedback | | |
| My debit card still hasn't arrived. | Negative Feedback | | |
| What is the status of ticket 123456? | Query | | |
| Thanks, but my issue still is not fixed. | Discussion | | |

## 12. Evaluation

A working demo is not enough. You should also evaluate your system.

### Classification Evaluation

- Did the classifier choose the correct category?
- Which examples failed?
- Were ambiguous messages handled reasonably?

### Routing Evaluation

- Did the orchestrator send each message to the correct agent?

### Ticket Evaluation

- Were tickets created correctly?
- Were duplicate or missing tickets handled?

### Response Quality Evaluation

- Were responses clear?
- Were responses professional?
- Were negative feedback responses empathetic?

In [22]:
# Create your own evaluation test cases here.
# Add at least 5 more examples.

orchestrator = OrchestratorAgent(
    classifier=ClassifierAgent(),
    feedback_agent=FeedbackAgent(),
    query_agent=QueryAgent(),
)
evaluation_cases = [
    # {"message": "...", "expected_category": "..."},
    "Thank you for helping me regain access to my net banking account.",

    "I reported this issue last week, but my credit card replacement still hasn't arrived.",

    "Could you please check the status of ticket 650932?",

    "The mobile banking app keeps crashing whenever I try to transfer money.",

    "Can you tell me whether ticket 784521 has been resolved yet?"
]

for msg in evaluation_cases:
    print("User:", msg)
    print("Assistant:", orchestrator.handle_message(msg))
    print("Classifier:", classifier.classify(msg))
    print("-" * 60)

import json
import pandas as pd

with open("logs.json", "r") as f:
    logs = json.load(f)

# Display as a table
pd.DataFrame(logs)

User: Thank you for helping me regain access to my net banking account.
Assistant: {'classification': 'Positive Feedback', 'route': 'Positive Feedback', 'response': "Dear Customer, thank you for your kind feedback! We're delighted to have helped you regain access to your net banking account and are always here to assist you.", 'ticket_number': None}
Classifier: Positive Feedback
------------------------------------------------------------
User: I reported this issue last week, but my credit card replacement still hasn't arrived.
Assistant: {'classification': 'Negative Feedback', 'route': 'Negative Feedback', 'response': 'Dear Customer, thank you for your patience regarding your credit card replacement; we apologize for the delay and are actively looking into your case with ticket number 551845 to resolve this as quickly as possible.', 'ticket_number': '551845'}
Classifier: Negative Feedback
------------------------------------------------------------
User: Could you please check the st

,timestamp,message,classification,route,response
0,2026-06-13T00:06:22.388634,Thanks for resolving my issue.,Positive Feedback,Positive Feedback,"Dear Customer, thank you for your kind feedbac..."
1,2026-06-13T00:06:23.715689,My debit card still hasn't arrived.,Negative Feedback,Negative Feedback,"Dear Customer, we apologize for the delay in r..."
2,2026-06-13T00:06:24.446556,What is the status of ticket 123456?,Query,Query,Ticket #123456 could not be found.
3,2026-06-13T00:06:25.876068,"Thanks, but my issue still is not fixed.",Negative Feedback,Negative Feedback,"Dear Customer, thank you for your feedback. We..."
4,2026-06-13T00:06:30.387194,Thank you for helping me regain access to my n...,Positive Feedback,Positive Feedback,"Dear Customer, thank you for your kind feedbac..."
5,2026-06-13T00:06:32.864442,"I reported this issue last week, but my credit...",Negative Feedback,Negative Feedback,"Dear Customer, thank you for your patience reg..."
6,2026-06-13T00:06:34.161715,Could you please check the status of ticket 65...,Query,Query,Ticket #650932 could not be found.
7,2026-06-13T00:06:36.852337,The mobile banking app keeps crashing whenever...,Negative Feedback,Negative Feedback,"Dear Customer, we apologize for the inconvenie..."
8,2026-06-13T00:06:38.986567,Can you tell me whether ticket 784521 has been...,Query,Query,Ticket #784521 could not be found.


## 13. Stretch Goals

If you finish early, choose one improvement:

- Add a Streamlit UI
- Use SQLite instead of JSON file storage
- Add logs showing classifier output and route taken
- Add a confidence score to classification
- Build a LangGraph version
- Build a CrewAI version
- Add a new specialized agent, such as Credit Card Agent or Loan Agent

In [24]:
!pip install streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 41.1 MB/s eta 0:00:00


## 13.1. Stretch Goals
- Add logs showing classifier output and route taken
- Add a confidence score to classification
- Build a LangGraph version


Streamlit UI

In [25]:
import streamlit as st
import pandas as pd
import json

st.set_page_config(
    page_title="Banking Customer Support AI",
    layout="wide"
)

st.title("🏦 Banking Customer Support AI Agent")

user_message = st.text_area(
    "Enter Customer Message"
)

if st.button("Submit"):

    result = orchestrator.run(user_message)

    st.subheader("Classification")
    st.success(result["classification"])

    st.subheader("Agent Route")
    st.info(result["route"])

    st.subheader("Response")
    st.write(result["response"])

    if result.get("ticket_number"):
        st.subheader("Generated Ticket")
        st.write(result["ticket_number"])

st.divider()

st.header("Support Tickets")

try:
    tickets = pd.read_json("tickets.json")
    st.dataframe(tickets)
except:
    st.warning("No tickets found.")

2026-06-13 00:07:26.408 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-13 00:07:26.409 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-13 00:07:26.563 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-06-13 00:07:26.564 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-13 00:07:26.565 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-13 00:07:26.567 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-13 00:07:26.567 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [26]:
%%writefile app.py
import streamlit as st
import pandas as pd
import json
# from google.colab import userdata # Removed as it causes issues in non-Colab environments
import os # Added for environment variable access
import random # Added for FeedbackAgent
from pathlib import Path # Added for FeedbackAgent
from openai import OpenAI

# Assuming these classes are defined in previous cells and are accessible
# Re-define them here or ensure they are imported/accessible in the Streamlit context
# For simplicity, let's assume they are already defined or we'll define minimal versions for the file

# --- Minimal Definitions (if not already globally accessible in the execution environment) ---
# These are placeholder classes to allow `app.py` to be written without errors.
# In a real scenario, these would need to be properly imported or defined within app.py

# Placeholder ClassifierAgent (ensure the actual class is available)
class ClassifierAgent:
    def __init__(self, model="gpt-3.5-turbo"):
        self.model = model
        # Retrieve the API key from environment variables
        api_key = os.environ.get('OPENAI_API_KEY')
        self.client = OpenAI(api_key=api_key) # Initialize the OpenAI client with the API key

    def classify(self, message: str) -> str:
        prompt = f"""
You are a banking customer support message classifier.

Classify the customer message into exactly one category:
- Positive Feedback
- Negative Feedback
- Query

Definitions:
- Positive Feedback: Appreciation, gratitude, praise, or satisfaction.
- Negative Feedback: Complaints, dissatisfaction, unresolved issues, frustration, or service problems.
- Query: Requests for information, ticket status checks, or questions.

Return ONLY the category name.
Do not provide explanations.

Examples:
"Thanks for your help" -> Positive Feedback
"My issue is still unresolved" -> Negative Feedback
"What is the status of ticket 123456?" -> Query

Customer Message:
"{message}"
"""

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        category = response.choices[0].message.content.strip()

        valid_categories = [
            "Positive Feedback",
            "Negative Feedback",
            "Query"
        ]

        if category not in valid_categories:
            return "Negative Feedback"

        return category


# Placeholder FeedbackAgent (ensure the actual class is available)
class FeedbackAgent:
    def __init__(self, ticket_file: str = "tickets.json"):
        self.ticket_file = Path(ticket_file)
        # Note: In a real Streamlit app, you might need to handle
        # persistent storage differently, e.g., using st.session_state
        # or ensuring the file path is accessible in the container.
        self._initialize_ticket_file()
        # self.client = OpenAI(api_key=userdata.get('OPENAI_API_KEY')) # Old line
        self.client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY')) # New line

    def _initialize_ticket_file(self):
        if not self.ticket_file.exists():
            self.ticket_file.write_text("[]")

    def _load_tickets(self) -> list[dict]:
        if not self.ticket_file.exists():
            return []
        content = self.ticket_file.read_text()
        if not content: # Handle empty file case
            return []
        try:
            return json.loads(content)
        except json.JSONDecodeError:
            return [] # Return empty list if JSON is invalid

    def _save_tickets(self, tickets: list[dict]) -> None:
        self.ticket_file.write_text(json.dumps(tickets, indent=2))

    def _generate_unique_ticket_number(self) -> str:
        tickets = self._load_tickets()

        existing_ticket_numbers = {ticket.get("ticket_number") for ticket in tickets if isinstance(ticket, dict) and "ticket_number" in ticket}

        while True:
            ticket_number = str(random.randint(100000, 999999))
            if ticket_number not in existing_ticket_numbers:
                return ticket_number

    def _create_ticket(self, customer_name: str, message: str) -> str:
        ticket_number = self._generate_unique_ticket_number()

        tickets = self._load_tickets()
        tickets.append({
            "ticket_number": ticket_number,
            "customer_name": customer_name,
            "message": message,
            "status": "Unresolved"
        })
        self._save_tickets(tickets)

        return ticket_number

    def _generate_response(
        self,
        feedback_type: str,
        customer_name: str,
        message: str,
        ticket_number: str | None = None
    ) -> str:
        prompt = f"""
You are a banking customer support assistant.

Generate a response for this customer.

Feedback Type: {feedback_type}
Customer Name: {customer_name}
Customer Message: {message}
Ticket Number: {ticket_number if ticket_number else "None"}

Requirements:
- Generate a warm and professional response
- Keep the response to 1-2 sentences.
- Do not invent account details.
- If a ticket number is provided, include it.
"""

        # Using client.chat.completions.create for consistency with ClassifierAgent
        response = self.client.chat.completions.create(
            model="gpt-3.5-turbo", # Assuming a general model for response generation
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.7,
        )

        return response.choices[0].message.content.strip()

    def handle_feedback(
        self,
        feedback_type: str,
        message: str,
        customer_name: str = "Customer"
    ) -> str:
        if feedback_type == "Positive Feedback":
            return self._generate_response(
                feedback_type=feedback_type,
                customer_name=customer_name,
                message=message
            )

        if feedback_type == "Negative Feedback":
            ticket_number = self._create_ticket(
                customer_name=customer_name,
                message=message
            )

            return self._generate_response(
                feedback_type=feedback_type,
                customer_name=customer_name,
                message=message,
                ticket_number=ticket_number
            )

        return "Invalid feedback type."


# Placeholder QueryAgent (ensure the actual class is available)
import re
# import os # Already moved to top

class QueryAgent:
    def __init__(self, tickets_file="tickets.json"):
        self.tickets_file = tickets_file

        if not os.path.exists(self.tickets_file):
            with open(self.tickets_file, "w") as file:
                json.dump([], file, indent=4)

    def extract_ticket_number(self, message):
        match = re.search(r"\b(\d{6})\b", message)

        if match:
            return match.group(1)

        return None

    def get_ticket_status(self, ticket_number):
        try:
            with open(self.tickets_file, "r") as file:
                tickets = json.load(file)

            # Assuming tickets is a list of dictionaries with 'ticket_number' key
            for ticket in tickets:
                if str(ticket.get("ticket_number")) == ticket_number:
                    return ticket.get("status", "Unknown")

        except (FileNotFoundError, json.JSONDecodeError):
            return None

        return None

    def handle_query(self, message):
        ticket_number = self.extract_ticket_number(message)

        if not ticket_number:
            return (
                "Could you please provide your 6-digit ticket number "
                "so that I can check the status for you?"
            )

        status = self.get_ticket_status(ticket_number)

        if status:
            return (
                f"Your ticket #{ticket_number} is currently "
                f"marked as: {status}."
            )

        return f"Ticket #{ticket_number} could not be found."


# Placeholder OrchestratorAgent (ensure the actual class is available)
class OrchestratorAgent:
    def __init__(self, classifier, feedback_agent, query_agent):
        self.classifier = classifier
        self.feedback_agent = feedback_agent
        self.query_agent = query_agent

    def handle_message(self, message: str) -> dict:
        category = self.classifier.classify(message)
        response_text = ""
        ticket_number = None
        route = category # For display purposes

        if category in ["Positive Feedback", "Negative Feedback"]:
            response_text = self.feedback_agent.handle_feedback(category, message)
            if category == "Negative Feedback":
                # Attempt to extract ticket number if generated by feedback agent
                match = re.search(r'ticket number (\d{6})', response_text)
                if match:
                    ticket_number = match.group(1)
        elif category == "Query":
            response_text = self.query_agent.handle_query(message)
        else:
            response_text = (
                "I'm sorry, I could not determine how to handle your request. "
                "Please rephrase your message."
            )
            route = "Unknown"

        return {"classification": category, "route": route, "response": response_text, "ticket_number": ticket_number}

# --- End of Minimal Definitions ---

# Initialize agents (ensure these match how they are initialized in the notebook)
# For Streamlit, it's often good practice to initialize these once, e.g., using st.session_state
if 'orchestrator' not in st.session_state:
    st.session_state.classifier_agent = ClassifierAgent()
    st.session_state.feedback_agent = FeedbackAgent()
    st.session_state.query_agent = QueryAgent()
    st.session_state.orchestrator = OrchestratorAgent(
        classifier=st.session_state.classifier_agent,
        feedback_agent=st.session_state.feedback_agent,
        query_agent=st.session_state.query_agent,
    )


st.set_page_config(
    page_title="Banking Customer Support AI",
    layout="wide"
)

st.title("🏦 Banking Customer Support AI Agent")

user_message = st.text_area(
    "Enter Customer Message"
)

if st.button("Submit"):
    if user_message:
        # Call the orchestrator's handle_message method
        result = st.session_state.orchestrator.handle_message(user_message)

        st.subheader("Classification")
        st.success(result["classification"])

        st.subheader("Agent Route")
        st.info(result["route"])

        st.subheader("Response")
        st.write(result["response"])

        if result.get("ticket_number"):
            st.subheader("Generated Ticket")
            st.write(result["ticket_number"])
    else:
        st.warning("Please enter a message.")

st.divider()

st.header("Support Tickets")

# Ensure the ticket file exists and has content before trying to read
ticket_file_path = Path("tickets.json")
if ticket_file_path.exists() and ticket_file_path.read_text().strip():
    try:
        # Ensure the JSON is in a list format for pd.read_json
        # If it's a dict of tickets, convert it to a list of dicts.
        tickets_content = ticket_file_path.read_text()
        parsed_tickets = json.loads(tickets_content)

        if isinstance(parsed_tickets, dict): # If tickets are stored as a dict with ticket_number as key
            ticket_list = []
            for num, details in parsed_tickets.items():
                ticket_data = {"ticket_number": num}
                ticket_data.update(details)
                ticket_list.append(ticket_data)
            tickets_df = pd.DataFrame(ticket_list)
        elif isinstance(parsed_tickets, list): # If tickets are stored as a list of dicts
            tickets_df = pd.DataFrame(parsed_tickets)
        else:
            st.warning("Tickets file is not in a recognized format (dict or list).")
            tickets_df = pd.DataFrame()

        if not tickets_df.empty:
            st.dataframe(tickets_df)
        else:
            st.warning("No tickets found in file.")

    except json.JSONDecodeError:
        st.warning("Error decoding tickets.json. File might be corrupted.")
    except Exception as e:
        st.error(f"An unexpected error occurred while loading tickets: {e}")
else:
    st.warning("No tickets file found or file is empty.")

Writing app.py


To run the Streamlit app, execute the following cell. It will provide a public URL. Click on the URL to interact with your Streamlit application.

_Note: You might encounter a `ModuleNotFoundError` if any of the agent classes (`ClassifierAgent`, `FeedbackAgent`, `QueryAgent`, `OrchestratorAgent`) are not defined globally or imported correctly within the `app.py` script. The `%%writefile` command creates `app.py` as an independent script. I've added minimal definitions for them in `app.py` to ensure it's self-contained. Also, the `tickets.json` file needs to be present in the same directory where `app.py` is run for the `FeedbackAgent` and `QueryAgent` to function correctly._

In [27]:
# Get the OpenAI API key from Colab's user data
from google.colab import userdata
openai_api_key_value = userdata.get('OPENAI_API_KEY')

# Install localtunnel and run the Streamlit app with the API key as an environment variable
# The '&' runs the Streamlit app in the background
# The 'npx localtunnel --port 8501' exposes the Streamlit app to a public URL
!npm install localtunnel
!export OPENAI_API_KEY=$openai_api_key_value && streamlit run app.py & npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
added 22 packages in 2s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴⠙

your url is: https://hungry-tigers-dress.loca.lt
2026-06-13 00:07:48.328 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.181.233.207:8501

  Stopping...
^C


## LangGraph Version



In [28]:
pip install langgraph langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 6.2 MB/s eta 0:00:00


## LangGraph Version

-- Define State

-- Define Nodes



In [29]:
from typing import TypedDict


class AgentState(TypedDict):
    message: str
    classification: str
    response: str

def classify_node(state):
    # classifier needs to be initialized in the global scope or passed in
    global classifier # Ensure classifier is accessible
    result = classifier.classify(
        state["message"]
    )

    # The classify method returns a string directly, not a dictionary with a 'category' key
    state["classification"] = result

    return state

def feedback_node(state):
    # feedback_agent needs to be initialized in the global scope or passed in
    global feedback_agent # Ensure feedback_agent is accessible

    state["response"] = feedback_agent.handle_feedback(
        state["classification"], # Pass feedback_type first
        state["message"]
    )

    return state

def query_node(state):
    # query_agent needs to be initialized in the global scope or passed in
    global query_agent # Ensure query_agent is accessible

    state["response"] = query_agent.handle_query(
        state["message"]
    )

    return state

In [30]:
# Conditional routing

def route(state):

    if state["classification"] == "Query":
        return "query"

    return "feedback"

In [31]:
# Build Langgraph

from langgraph.graph import (
    StateGraph,
    END
)

# Initialize agents globally for Langgraph nodes to access them
# Assuming ClassifierAgent, FeedbackAgent, QueryAgent classes are defined in previous cells
global classifier, feedback_agent, query_agent
classifier = ClassifierAgent()
feedback_agent = FeedbackAgent()
query_agent = QueryAgent()

workflow = StateGraph(AgentState)

workflow.add_node(
    "classifier",
    classify_node
)

workflow.add_node(
    "feedback",
    feedback_node
)

workflow.add_node(
    "query",
    query_node
)

workflow.set_entry_point(
    "classifier"
)

workflow.add_conditional_edges(
    "classifier",
    route,
    {
        "feedback": "feedback",
        "query": "query"
    }
)

workflow.add_edge(
    "feedback",
    END
)

workflow.add_edge(
    "query",
    END
)

graph = workflow.compile()

In [32]:
# Execute Langgraph
result = graph.invoke(
    {
        "message":
        "What is the status of ticket 650932?"
    }
)

print(result["response"])

Ticket #650932 could not be found.


In [33]:
# End-to-End Test Cases
end_to_end_tests = [
    "Thank you for helping me regain access to my net banking account.",
    "I reported this issue last week, but my credit card replacement still hasn't arrived.",
    "Could you please check the status of ticket 650932?",
    "The mobile banking app keeps crashing whenever I try to transfer money.",
    "Can you tell me whether ticket 784521 has been resolved yet?"
]

# Execute the graph for each test case
results = []

for i, message in enumerate(end_to_end_tests, start=1):

    print("=" * 80)
    print(f"TEST CASE {i}")
    print("=" * 80)
    print(f"Customer Message: {message}")

    try:
        result = graph.invoke(
            {
                "message": message,
                "classification": "",
                "response": ""
            }
        )

        results.append(result)

        print(f"Classification : {result.get('classification')}")
        print(f"Response       : {result.get('response')}")

        if "ticket_number" in result:
            print(f"Ticket Number  : {result['ticket_number']}")

    except Exception as e:
        print(f"Execution Failed: {e}")

    print("\n")

TEST CASE 1
Customer Message: Thank you for helping me regain access to my net banking account.
Classification : Positive Feedback
Response       : Dear Customer, thank you for your kind words! We're delighted to have helped you regain access to your net banking account and are always here to assist you.


TEST CASE 2
Customer Message: I reported this issue last week, but my credit card replacement still hasn't arrived.
Classification : Negative Feedback
Response       : Dear Customer, we sincerely apologize for the delay in your credit card replacement and appreciate your patience; we are actively reviewing ticket number 973230 to resolve this promptly. Please rest assured, we will update you as soon as possible.


TEST CASE 3
Customer Message: Could you please check the status of ticket 650932?
Classification : Query
Response       : Ticket #650932 could not be found.


TEST CASE 4
Customer Message: The mobile banking app keeps crashing whenever I try to transfer money.
Classificatio

## Final Reflection

Answer these questions:

1. Which component was easiest to build?
2. Which component was hardest to build?
3. Which prompt improved the most after revision?
4. Which part did not need AI?
5. What would you improve in a real banking production system?

### Key Takeaway

Good AI systems are not just prompts. They are well-designed workflows with clear responsibilities, reliable non-AI components, testing, and evaluation.

In [34]:
# Final reflection notes

#The Query Agent was the easiest to build because it primarily relied on regular expressions to extract ticket numbers and simple JSON lookups to retrieve ticket statuses.
#The Classifier Agent was the hardest to build because designing prompts that consistently distinguished between positive feedback, negative feedback, and queries required multiple iterations and careful testing.
#The classification prompt improved the most after revision, as adding clearer definitions, examples, and strict output constraints significantly increased routing accuracy and reduced ambiguous responses.
#Ticket generation, ticket storage, status retrieval, and logging did not require AI, as these tasks were more reliably handled using deterministic Python logic and JSON-based data management.
#In a production environment, I would integrate secure databases, authentication mechanisms, human escalation workflows, monitoring dashboards, and more rigorous evaluation pipelines to improve reliability, security, and compliance.

#Key Takeaway
#Good AI systems are not just prompts. They combine well-designed workflows, clear agent responsibilities, robust non-AI components, comprehensive testing, and continuous evaluation to deliver reliable and trustworthy outcomes.